<a href="https://colab.research.google.com/github/dhag/colab_demo/blob/main/WS_echo_server_colab_ngrok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WebSocket エコーサーバ（受信側そのまま） / Colab + ngrok

あなたの `websockets` 製サーバを **ロジックそのまま**で Colab 上で動かし、ngrok で公開します。
クライアントはアップロード済みの HTML（`_14_web_sock_B.html`）をブラウザで開いて使います。

## 元コードから変えた点（2点だけ）
1. Colab には既に動いているイベントループがあるため `asyncio.run(main())` が使えない
   → **専用イベントループを別スレッドで回す**形にした（起動/停止関数つき）。
2. 公開は **ngrok の TCP トンネル**。HTML の url 欄はこれまで通り `ws://ホスト:ポート` のままでよい。

`echo` ハンドラ（numpy 処理・JSON 応答・バイナリエコーの順序）は**完全に元のまま**なので、
HTML クライアントが期待する `type:"text"` / `image_info` と画像エコーの動作はそのまま一致します。

実行順：① インストール → ② echo 定義 → ③ 起動 → ④ ngrok 公開（出た URL を HTML に貼る）。停止は⑤。

## ① インストール

In [ ]:
!pip install -q websockets pyngrok numpy

## ② echo ハンドラ
ここは元のコードを一切変えていません。

In [ ]:
import asyncio
import json
import base64
import numpy as np
import websockets

HOST = "127.0.0.1"
PORT = 8765

async def echo(websocket):
    print("client connected")
    try:
        async for message in websocket:
            # テキストの場合
            if isinstance(message, str):
                print("text recv:", message)
                codes = np.array([ord(c) for c in message], dtype=np.float64)
                mean_code = float(np.mean(codes)) if len(codes) > 0 else 0.0
                response = {
                    "type": "text",
                    "original": message,
                    "length": len(message),
                    "numpy_mean_code": mean_code,
                    "reply": "echo: " + message
                }
                await websocket.send(json.dumps(response, ensure_ascii=False))
            # バイナリの場合：画像など
            elif isinstance(message, bytes):
                print("binary recv:", len(message), "bytes")
                arr = np.frombuffer(message, dtype=np.uint8)
                mean_byte = float(np.mean(arr)) if arr.size > 0 else 0.0
                meta = {
                    "type": "image_info",
                    "bytes": len(message),
                    "numpy_mean_byte": mean_byte,
                    "note": "server received binary data and echoes it back"
                }
                await websocket.send(json.dumps(meta, ensure_ascii=False))
                # 次に画像本体をそのまま返す
                await websocket.send(message)
    except websockets.exceptions.ConnectionClosed:
        print("client disconnected")

## ③ サーバを起動（バックグラウンドの専用ループ）
`asyncio.run` の代わりに、別スレッドで専用イベントループを回します。
`websockets.serve(echo, HOST, PORT, max_size=20MB)` の呼び出し自体は元と同じ引数です。

In [ ]:
import threading, warnings, asyncio, time
warnings.filterwarnings("ignore", category=DeprecationWarning)

_loop = None
_stop = None
_thread = None

async def _amain():
    global _stop
    _stop = asyncio.get_running_loop().create_future()    # 停止用Future（ループ内で生成）
    async with websockets.serve(echo, HOST, PORT, max_size=20 * 1024 * 1024):
        print(f"WebSocket server running: ws://{HOST}:{PORT}")
        await _stop                                        # set_result されるまで待機
    print("server loop exited")

def _run():
    global _loop
    try:
        _loop = asyncio.new_event_loop()
        asyncio.set_event_loop(_loop)
        _loop.run_until_complete(_amain())                 # serve はコルーチン内で await する
        _loop.close()
    except Exception as e:
        print("‼ server thread error:", repr(e))

def start_server():
    global _thread
    if _thread and _thread.is_alive():
        print("already running（先に stop_server() を）"); return
    _thread = threading.Thread(target=_run, daemon=True, name="ws-server")
    _thread.start()
    time.sleep(0.5)
    print("alive:", _thread.is_alive())                    # True なら起動成功

def stop_server():
    global _thread
    if not (_thread and _thread.is_alive()):
        print("not running"); return
    if _stop and not _stop.done():
        _loop.call_soon_threadsafe(_stop.set_result, None)
    _thread.join(timeout=5)
    print("server stopped (port released)")

start_server()

WebSocket server running: ws://127.0.0.1:8765
alive: True


## ④(A) ngrok で公開（HTTP トンネル）
authtoken 入力後、出てきた **`wss://...` を HTML の url 欄に貼り付け**て Connect します。

In [ ]:
import getpass
from pyngrok import ngrok, conf

print("ngrok authtoken を貼り付け（https://dashboard.ngrok.com/get-started/your-authtoken ）")
conf.get_default().auth_token = getpass.getpass()

# HTTP トンネル = 443番(https/wss)経由。高ポートを塞ぐ学内FWでも通りやすい
tunnel = ngrok.connect(PORT, "http")
public = tunnel.public_url                      # 例: https://xxxx.ngrok-free.app
ws_url = "wss://" + public.split("://", 1)[1]   # → wss://xxxx.ngrok-free.app
print("=" * 56)
print("HTML の url 欄にこれを貼り付け:")
print("   ", ws_url)
print("=" * 56)

ngrok authtoken を貼り付け（https://dashboard.ngrok.com/get-started/your-authtoken ）
··········
HTML の url 欄にこれを貼り付け:
    wss://1b93-35-254-112-53.ngrok-free.app


## ④(B)  **ngrok** で公開（TCP トンネル）岩手大学のネットワークからはアクセス不可
authtoken 入力後、出てきた **`ws://...` を HTML の url 欄に貼り付け**て Connect します。

In [ ]:
import getpass
from pyngrok import ngrok, conf

print("ngrok authtoken を貼り付け（https://dashboard.ngrok.com/get-started/your-authtoken ）")
conf.get_default().auth_token = getpass.getpass()

tunnel = ngrok.connect(PORT, "tcp")              # 生TCPで8765を公開（HTTP割り込みページなし）
host_port = tunnel.public_url.split("://", 1)[1] # 例: 0.tcp.ngrok.io:12345
ws_url = "ws://" + host_port
print("=" * 56)
print("HTML の url 欄にこれを貼り付け:")
print("   ", ws_url)
print("=" * 56)

ngrok authtoken を貼り付け（https://dashboard.ngrok.com/get-started/your-authtoken ）
··········
HTML の url 欄にこれを貼り付け:
    ws://6.tcp.ngrok.io:28666


トラブルシューティング用。ブラウザを介さず Colab 内からサーバとトンネルを叩く診断セル。local OK / ngrok OK なら、サーバもトンネルも正常で、原因はブラウザ側の経路（＝高ポート遮断や mixed-content）

In [ ]:
import websockets

# (1) サーバ自身（ループバック）
try:
    async with websockets.connect("ws://127.0.0.1:8765") as ws:
        await ws.send("ping-local")
        print("local  OK :", await ws.recv())
except Exception as e:
    print("local  FAILED:", repr(e))

# (2) ngrok 経由（Colab → ngrok → サーバ。ブラウザ非経由）
try:
    async with websockets.connect(ws_url) as ws:   # wss:// のURL
        await ws.send("ping-ngrok")
        print("ngrok  OK :", await ws.recv())
except Exception as e:
    print("ngrok  FAILED:", repr(e))

## ⑤ 停止 / 再起動
止めるときは `stop_server()`。ポートも解放されます。
再起動は ③の `start_server()` → ④の ngrok を再実行（TCP ポートは毎回変わります）。

In [ ]:
stop_server()
# ngrok 側も閉じる場合:
# ngrok.disconnect(tunnel.public_url)   # 個別に閉じる
# ngrok.kill()                          # ngrok を完全停止

server loop exited
server stopped (port released)


---
### ブラウザ側の使い方（アップロード済み HTML）
1. `_14_web_sock_B.html` をブラウザで開く（ローカルの file:// で可）。
2. url 欄の `ws://127.0.0.1:8765` を、④で表示された `ws://0.tcp.ngrok.io:xxxxx` に置き換える。
3. **Connect** → テキスト送信や画像送信を試す。
   - テキスト → サーバが JSON（`reply` / `numpy_mean_code`）を返す。
   - 画像 → まず JSON メタ（`bytes` / `numpy_mean_byte`）、続けて画像本体がエコーされる。

### メモ
- TCP トンネルなので URL は `ws://`（`wss://` ではない）。HTML をローカルの file:// で開けば
  非セキュアな `ws://` 接続もブラウザに許可される。
- 接続できないときの主因は authtoken の誤り、または③のサーバ未起動。④を再実行すると新しい URL が出る。
- TCP デモ（長さヘッダ自作）と違い、WebSocket は1メッセージが丸ごと届くのでフレーミング不要、という対比が見どころ。